# TX Strategy Research

## Environment

In [1]:
from datetime import datetime
from pathlib import Path
import os
import sys

import pandas as pd
from tvDatafeed import Interval, TvDatafeed

NOTE_DIR = Path.cwd().resolve().parents[3] / 'note'
sys.path.extend([str(NOTE_DIR), os.getcwd()])

%load_ext autoreload
%autoreload 2

from module.get_info_FinMind import FinMindClient
from module.get_info_TWSE import GetInfoTWSE
from module.options.option_tools import compute_iv
from analyzer import StrategyConfig, TXAnalyzer

START = '2020-01-01'
END = datetime.now().strftime('%Y-%m-%d')

TRAIN_RATIO = 0.7
VALIDATION_RATIO = 0.05
SETTLEMENT_PATH = NOTE_DIR / 'data' / 'settle_TXO.csv'

tv = TvDatafeed()
fm = FinMindClient()
twse = GetInfoTWSE()

you are using nologin method, data you access may be limited
2026-07-18 17:56:03.312 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success


## Base Market Data

In [2]:
fm.initialize_frame(stock_id='TX', start_time=START, end_time=END)
analyzer = TXAnalyzer(fm.get_future_price())
analyzer.session_alignment_report()

split_dates = TXAnalyzer.split_periods(
    analyzer.display_df().index,
    start=StrategyConfig().backtest_start_date,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
)
TRAIN_END = split_dates['train_end']
VALIDATION_START = split_dates['validation_start']
VALIDATION_END = split_dates['validation_end']
TEST_START = split_dates['test_start']
pd.Series(split_dates, name='date')

train_end          2026-06-05
validation_start   2026-06-08
validation_end     2026-06-11
test_start         2026-06-12
Name: date, dtype: datetime64[ns]

## Factor Discovery

每個小節都是完整的探索單元：取得資料、對齊資料、畫圖或檢查關係。Discovery 一律只使用 training 資料，不會查看 validation 或 test 期間。


### Price and Calendar

In [5]:
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.daily_ret()
training_analyzer.monthly_ret(mode='benchmark')
training_analyzer.indicator_weekday_stats()
training_analyzer.indicator_night_price()
training_analyzer.indicator_gap_days(after_holiday=False)
training_analyzer.indicator_gap_days(after_holiday=True)


In [7]:
# ma 參數
for w in range(5, 46, 5):
    training_analyzer.indicator_15ma_divergence(window=w)

In [33]:
training_analyzer.compare_ma_divergence_windows(
    range(5, 51, 1),
    bin_percentile=4,
    demean_return=False,
)

training_analyzer.compare_ma_divergence_windows(
    range(5, 51, 1),
    bin_percentile=4,
    demean_return=False,
    volatility_window=20,
    volatility_bins=5,
)

,observations,minimum_return,divergence_at_minimum
window,,,
5,1461,-0.053168,-0.048622
6,1461,-0.053168,-0.065963
7,1461,-0.053168,-0.080100
8,1461,-0.053168,-0.090045
9,1461,-0.053168,-0.100738
10,1461,-0.053168,-0.111019
11,1461,-0.053168,-0.118283
12,1461,-0.053168,-0.123970
13,1461,-0.053168,-0.127778


In [34]:
training_analyzer.compare_ma_divergence_by_volatility(
    range(5, 51, 1),
    return_column='daily_ret',
    volatility_window=20,
    volatility_bins=5,
    divergence_bin_percentile=20,
)

volatility_regime         Q1 low vol                                      \
divergence_percentile_bin  0% to 20% 20% to 40%    40% to 60% 60% to 80%   
5                           0.000375  -0.000845  6.414669e-04   0.001767   
6                          -0.000151  -0.000590  1.141476e-03   0.001220   
7                           0.000190  -0.001185  1.768214e-03   0.000679   
8                           0.000506  -0.000939  9.540076e-04   0.001494   
9                           0.000392  -0.000455  2.978729e-04   0.001553   
10                          0.000322  -0.000506  4.953767e-04   0.001452   
11                          0.000322  -0.000717  8.250457e-04   0.001372   
12                          0.000281  -0.000891  9.413786e-04   0.001400   
13                          0.000072  -0.000805  9.227446e-04   0.001490   
14                          0.000132  -0.000625  4.274939e-04   0.001865   
15                          0.000054  -0.000665  6.377736e-04   0.001831   
16                         -0.000108  -0.000638  9.540354e-04   0.001628   
17                          0.000182  -0.000884  9.402255e-04   0.001653   
18                          0.000182  -0.000859  8.095807e-04   0.001501   
19                         -0.000128  -0.000424  2.031463e-04   0.001925   
20                          0.000221  -0.001038  3.787654e-04   0.001751   
21                          0.000234  -0.000863 -3.091962e-04   0.001967   
22                          0.000298  -0.001028 -1.063653e-04   0.001765   
23                          0.000282  -0.001070  2.589493e-04   0.001645   
24                          0.000596  -0.001136  8.290289e-05   0.001469   
25                          0.000282  -0.000716 -5.732002e-05   0.001497   
26                          0.000330  -0.000644 -1.174112e-04   0.001525   
27                          0.000120  -0.000413 -2.221308e-04   0.001657   
28                         -0.000049  -0.000428 -9.803873e-05   0.001435   
29                         -0.000049  -0.000387 -1.382796e-04   0.001349   
30                         -0.000049  -0.000489 -3.038236e-04   0.001760   
31                         -0.000037  -0.000700 -5.856511e-05   0.001649   
32                         -0.000271  -0.000543 -7.660477e-07   0.001704   
33                         -0.000174  -0.000474  9.192795e-05   0.001537   
34                         -0.000435  -0.000086  1.738286e-04   0.001545   
35                         -0.000049  -0.000336  6.421759e-04   0.001266   
36                         -0.000183   0.000028  2.057667e-04   0.001326   
37                         -0.000608   0.000599  1.574604e-04   0.000697   
38                         -0.000608   0.000677  2.636162e-04   0.000755   
39                         -0.000608   0.000677  1.646908e-04   0.000854   
40                         -0.000608   0.000735  4.022122e-05   0.000767   
41                         -0.000667   0.000794 -4.226557e-05   0.000849   
42                         -0.000667   0.000712 -1.216358e-04   0.001010   
43                         -0.000667   0.000618 -2.726842e-05   0.001220   
44                         -0.000803   0.000728  4.166728e-05   0.001048   
45                         -0.000803   0.000549  3.480626e-04   0.000921   
46                         -0.000688   0.000321  4.901787e-04   0.000834   
47                         -0.000371   0.000218  5.680197e-04   0.000527   
48                         -0.000611   0.000305  8.802143e-04   0.000367   
49                         -0.000217  -0.000089  9.583870e-04   0.000305   
50                         -0.000257  -0.000077  7.601475e-04   0.000259   

volatility_regime                            Q2                        \
divergence_percentile_bin 80% to 100% 0% to 20% 20% to 40% 40% to 60%   
5                            0.000382  0.000384   0.000519   0.000050   
6                            0.000698  0.000458   0.000999   0.000108   
7                            0.000884  0.000682   0.000934 

### Option Positioning

In [ ]:
raw_day = fm.option_institution_position('TXO', start_date=START, end_date=END).copy()
raw_night = twse.get_institution_option_position(
    trading_session='night',
    start_time=START,
    end_time=END,
).copy()
analyzer.add_option_signals(raw_day, raw_night)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_opt_position(
    indicator='Foreign_Opt_Signal',
    trading_session='day',
    sub_analysis=False,
)

### Option Implied Volatility

In [ ]:
option_daily = TXAnalyzer.fetch_option_daily(
    fm,
    option_id='TXO',
    start_date=START,
    end_date=END,
)
settlement_df = pd.read_csv(SETTLEMENT_PATH)
analyzer.add_option_iv_skew(
    option_daily,
    settlement_df,
    iv_calculator=compute_iv,
    risk_free_rate=0.015,
)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_option_iv(trading_session='day')

### Sentiment

In [ ]:
historical_fear_greed = fm.get_cnn_fear_greed(START, END)
latest_fear_greed = TXAnalyzer.fetch_fear_greed()
analyzer.add_fear_greed(historical_fear_greed, latest_fear_greed)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_fear_greed(trading_session='day')
training_analyzer.indicator_fear_greed(trading_session='night')

### Global Market Factors

In [ ]:
move = tv.get_hist(
    symbol='MOVE',
    exchange='TVC',
    interval=Interval.in_daily,
    n_bars=3000,
)
analyzer.add_market_ohlc(move, 'MOVE')
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_move(trading_session='day')
training_analyzer.indicator_move(trading_session='night')
training_analyzer.indicator_move(trading_session='day', sub_analysis=True)

sox = tv.get_hist(
    symbol='SOX',
    exchange='TVC',
    interval=Interval.in_daily,
    n_bars=3000,
)
analyzer.add_market_ohlc(sox, 'SOX')
training_analyzer.indicator_sox(trading_session='day')

### Rates and Margin

In [ ]:
bond_5_year = fm.get_US_bond('United States 5-Year', START, END)
bond_features = bond_5_year[['date', 'value']].rename(columns={'value': 'US_bond_5y'}).set_index('date')
analyzer.merge_features(bond_features)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_US_bond(indicator='yield_shock')

margin_maintenance = fm.get_total_margin_maintenance(start_time=START, end_time=END)
analyzer.merge_features(margin_maintenance.set_index('date'))
margin_balance = fm.get_total_margin_info()
margin_features = margin_balance.pivot_table(index='date', columns='name', values='TodayBalance')
analyzer.merge_features(margin_features)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_maintenance_rate(point_version=False)
training_analyzer.indicator_margin_delta()

## Research Data Check

確認這次實際跑過的研究資料是否已合併、是否有缺值，以及最後更新日。

In [ ]:
analyzer.for_period(end=TRAIN_END).feature_status([
    'MOVE_open',
    'SOX_open',
    'Foreign_Opt_Signal_a',
    'SkewSlope',
    'fear_greed',
    'US_bond_5y',
])

## Hypothesis and Position Design

只有通過探索、且能說明交易理由的關係才放進設定。這裡建立基準假設與要驗證的候選部位。

In [ ]:
baseline_config = StrategyConfig()

candidate_config = StrategyConfig(
    day_long_position=0.5,
    day_short_position=-0.5,
    night_position=0.0,
)

## Validation

只用 validation 期間比較已經提出的設定，選定一個設定後就不要再依 validation 或 test 結果修改它。


In [ ]:
baseline_validation = analyzer.evaluate(
    baseline_config,
    start=VALIDATION_START,
    end=VALIDATION_END,
)
candidate_validation = analyzer.evaluate(
    candidate_config,
    start=VALIDATION_START,
    end=VALIDATION_END,
)

validation_summary = pd.concat(
    {
        'baseline': analyzer.summarize_result(baseline_validation),
        'candidate': analyzer.summarize_result(candidate_validation),
    },
    axis=1,
)
validation_summary


In [ ]:
validation_comparison = pd.DataFrame({
    'baseline': baseline_validation['strat_ret'],
    'candidate': candidate_validation['strat_ret'],
}).dropna()
validation_comparison.cumsum().plot(title='Validation: Baseline vs Candidate')


## Test

選定設定後才執行這段。Test 期間只用來確認固定策略的表現，不再調整設定。


In [ ]:
selected_config = baseline_config

test_result = analyzer.evaluate(selected_config, start=TEST_START)
test_summary = analyzer.summarize_result(test_result)
test_summary

# 確認 test 後，再將同一份固定設定寫入正式結果檔。
analyzer.backtest(config=selected_config, point_version=False, start=TEST_START)
